# GPU Sorting — Colab Runner

**Flux de lucru:**
1. `Runtime → Change runtime type → T4 GPU`
2. Rulează celula de verificare GPU
3. Rulează `zip_for_colab.sh` local, apoi uploadează `gpu_sorting_colab.zip`
4. Compilează cu `make`
5. Generează date de test (binar)
6. Rulează fiecare algoritm
7. Descarcă `results/timings.csv`

## 1. Verificare GPU

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Upload și unzip proiect

Rulează `zip_for_colab.sh` local, apoi uploadează `gpu_sorting_colab.zip`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # selectează gpu_sorting_colab.zip

In [ ]:
import zipfile, os

ZIP_NAME = "gpu_sorting_colab.zip"
WORK_DIR = "/content/gpu_sorting"

os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_NAME, 'r') as z:
    z.extractall(WORK_DIR)

os.makedirs(f"{WORK_DIR}/data",    exist_ok=True)
os.makedirs(f"{WORK_DIR}/results", exist_ok=True)
os.makedirs(f"{WORK_DIR}/bin",     exist_ok=True)

print("Unzip complet.")

## 3. Compilare cu nvcc

In [ ]:
import subprocess

result = subprocess.run(["make", "all"], capture_output=True, text=True, cwd=WORK_DIR)
print(result.stdout)
if result.returncode != 0:
    print("ERORI DE COMPILARE:")
    print(result.stderr)
else:
    print("Compilare reusita!")

## 4. Generare date de test (format binar)

Scrie `data/input.bin` în formatul citit de C: `[int n][n * int32]`.

In [ ]:
import numpy as np, struct

N = 1_000_000  # 1M elemente — mărește spre 10M dupa ce verifici ca merge

data = np.random.randint(0, 2**31 - 1, size=N, dtype=np.int32)

bin_path = f"{WORK_DIR}/data/input.bin"
with open(bin_path, 'wb') as f:
    f.write(struct.pack('i', N))   # primii 4 bytes = n
    data.tofile(f)                 # n * int32

print(f"Generat {N:,} elemente -> data/input.bin ({os.path.getsize(bin_path):,} bytes)")
print(f"Primele 10: {data[:10]}")

## 5. Rulare algoritmi

In [ ]:
algorithms = [
    ("Bitonic Sort",  "bin/bitonic_sort"),
    ("Shell Sort",    "bin/shell_sort"),
    ("Odd-Even Sort", "bin/odd_even_sort"),
    ("Ranking Sort",  "bin/ranking_sort"),
    ("Merge Sort",    "bin/merge_sort"),
]

# Sterge CSV anterior ca sa nu acumuleze date vechi
csv_path = f"{WORK_DIR}/results/timings.csv"
if os.path.exists(csv_path):
    os.remove(csv_path)

for name, binary in algorithms:
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print('='*60)
    proc = subprocess.run(
        [f"{WORK_DIR}/{binary}"],
        capture_output=True, text=True,
        cwd=WORK_DIR        # important: cale relativa data/ si results/ sa fie corecte
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(f"EROARE: {proc.stderr}")

## 6. Afisare si download rezultate

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
print(df.to_string(index=False))

In [ ]:
files.download(csv_path)